<a href="https://colab.research.google.com/github/Babukanth/Healthcare_Data_Cleaning/blob/main/Healthcare_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd
import numpy as np

data = {
    'patient_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
                   111, 112, 113, 114, 115, 101, 107, 118, 119, 120],
    'age': ['25', '34', None, '45', '29', None, '38', '52', '27', '41',
            '33', 'unknown', '48', '26', '35', '25', '38', '31', None, '44'],
    'weight': ['70', '65', '80', None, '75', None, '68', '90', '72', '85',
               '78', None, '82', '69', 'N/A', '70', '68', '74', None, '88'],
    'blood_pressure': [120, 130, None, 140, 125, None, 135, None, 118, 145,
                      128, None, 138, 122, None, 120, 135, 126, None, 142],
    'medication': ['Aspirin', 'Metformin', 'Lisinopril', None, 'Aspirin',
                   'Metformin', 'Lisinopril', 'Aspirin', None, 'Metformin',
                   'Lisinopril', 'Aspirin', None, 'Metformin', 'Aspirin',
                   'Aspirin', 'Lisinopril', 'Metformin', 'Aspirin', None],
    'insurance_provider': ['Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', None,
                          'Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', 'Blue Cross',
                          'Aetna', None, 'UnitedHealth', 'Blue Cross', 'Aetna',
                          'Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', None]
}

df = pd.DataFrame(data)
print(df)

    patient_id      age weight  blood_pressure  medication insurance_provider
0          101       25     70           120.0     Aspirin         Blue Cross
1          102       34     65           130.0   Metformin              Aetna
2          103     None     80             NaN  Lisinopril              Cigna
3          104       45   None           140.0        None       UnitedHealth
4          105       29     75           125.0     Aspirin               None
5          106     None   None             NaN   Metformin         Blue Cross
6          107       38     68           135.0  Lisinopril              Aetna
7          108       52     90             NaN     Aspirin              Cigna
8          109       27     72           118.0        None       UnitedHealth
9          110       41     85           145.0   Metformin         Blue Cross
10         111       33     78           128.0  Lisinopril              Aetna
11         112  unknown   None             NaN     Aspirin      

**1. Inspect the Data**


In [32]:
#Understanding the dataset before cleaning helps you choose the right strategies.
#Basic structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   patient_id          20 non-null     int64  
 1   age                 17 non-null     object 
 2   weight              16 non-null     object 
 3   blood_pressure      14 non-null     float64
 4   medication          16 non-null     object 
 5   insurance_provider  17 non-null     object 
dtypes: float64(1), int64(1), object(4)
memory usage: 1.1+ KB


In [33]:
#Percentage of missing values
(df.isnull().sum() / len(df)) * 100

,0
patient_id,0.0
age,15.0
weight,20.0
blood_pressure,30.0
medication,20.0
insurance_provider,15.0


In [34]:
#Duplicate rows
df.duplicated().sum()

np.int64(2)

**2. Data Type Conversion**

In [35]:
#The dataset contains numbers stored as strings and invalid entries like "unknown" and "N/A".
#Convert age to numeric
df['age'] = pd.to_numeric(df['age'], errors='coerce')
#print(df)

In [36]:
#Convert weight to numeric
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
#print(df)

In [37]:
#Count new missing values created
df.isnull().sum()


,0
patient_id,0
age,4
weight,5
blood_pressure,6
medication,4
insurance_provider,3


In [38]:
#Convert insurance_provider to category
df['insurance_provider'] = df['insurance_provider'].astype('category')
#print(df)

In [39]:
#Verify
df.dtypes

,0
patient_id,int64
age,float64
weight,float64
blood_pressure,float64
medication,object
insurance_provider,category


**3. Handling Missing Values**

In [40]:
#Each column needs a different strategy based on meaning and distribution.
#Fill age with median
df['age'] = df['age'].fillna(df['age'].median())


In [41]:
#Fill weight with median
df['weight'] = df['weight'].fillna(df['weight'].median())


In [42]:
#Fill blood_pressure with median
df['blood_pressure'] = df['blood_pressure'].fillna(df['blood_pressure'].median())


In [43]:
#Fill medication with mode
df['medication'] = df['medication'].fillna(df['medication'].mode()[0])


In [44]:
#Fill insurance_provider with constant
#Since the Unknown Category is not there, we have added the category before filling the value
df['insurance_provider'] = df['insurance_provider'].cat.add_categories('Unknown')

df['insurance_provider'] = df['insurance_provider'].fillna('Unknown')

In [45]:
#Verify no missing values remain
df.isnull().sum()


,0
patient_id,0
age,0
weight,0
blood_pressure,0
medication,0
insurance_provider,0


**4. Handling Duplicates**

In [46]:
#View duplicate rows
df[df.duplicated(keep=False)]


,patient_id,age,weight,blood_pressure,medication,insurance_provider
0,101,25.0,70.0,120.0,Aspirin,Blue Cross
6,107,38.0,68.0,135.0,Lisinopril,Aetna
15,101,25.0,70.0,120.0,Aspirin,Blue Cross
16,107,38.0,68.0,135.0,Lisinopril,Aetna


In [47]:
#Identify duplicates based on patient_id
df.duplicated(subset=['patient_id'])


,0
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,False
9,False


In [48]:
#Remove duplicates (keep first)
df_before = df.shape
df = df.drop_duplicates(subset=['patient_id'], keep='first')
df_after = df.shape
df_before, df_after


((20, 6), (18, 6))

**5. Full Cleaning Workflow (Start Fresh)**

In [49]:
#Reload original data
df = pd.DataFrame(data)
df_clean = df.copy()

In [50]:
#Step-by-step cleaning pipeline
#A. Data types
df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce')
df_clean['weight'] = pd.to_numeric(df_clean['weight'], errors='coerce')
df_clean['insurance_provider'] = df_clean['insurance_provider'].astype('category')

In [51]:
#B. Missing values
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['weight'] = df_clean['weight'].fillna(df_clean['weight'].median())
df_clean['blood_pressure'] = df_clean['blood_pressure'].fillna(df_clean['blood_pressure'].median())
df_clean['medication'] = df_clean['medication'].fillna(df_clean['medication'].mode()[0])
df_clean['insurance_provider'] = df_clean['insurance_provider'].cat.add_categories('Unknown')
df_clean['insurance_provider'] = df_clean['insurance_provider'].fillna('Unknown')

In [52]:
#C. Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['patient_id'], keep='first')

**Verification Report**

In [53]:
#Shape before and after
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)



Original shape: (20, 6)
Cleaned shape: (18, 6)


In [54]:
#Missing values before and after
print("Missing before:\n", df.isnull().sum())
print("Missing after:\n", df_clean.isnull().sum())


Missing before:
 patient_id            0
age                   3
weight                4
blood_pressure        6
medication            4
insurance_provider    3
dtype: int64
Missing after:
 patient_id            0
age                   0
weight                0
blood_pressure        0
medication            0
insurance_provider    0
dtype: int64


In [55]:
#Duplicates before and after
print("Duplicates before:", df.duplicated(subset=['patient_id']).sum())
print("Duplicates after:", df_clean.duplicated(subset=['patient_id']).sum())


Duplicates before: 2
Duplicates after: 0


In [56]:
#Data types before and after
print("Types before:\n", df.dtypes)
print("Types after:\n", df_clean.dtypes)


Types before:
 patient_id              int64
age                    object
weight                 object
blood_pressure        float64
medication             object
insurance_provider     object
dtype: object
Types after:
 patient_id               int64
age                    float64
weight                 float64
blood_pressure         float64
medication              object
insurance_provider    category
dtype: object
